# Resident Outcome Models — Training

Trains three models from `warehouse.fact_resident_outcomes_ml`:

1. **Reintegration outcome** — `Completed / In Progress / Not Started / On Hold`
2. **Reintegration type** — `Foster Care / Family Reunification / Independent Living / Adoption`
3. **Education completion** — `Completed / InProgress / NotStarted`

Outputs: 3 `.sav` files + metadata + metrics JSON files.

In [ ]:
import sys
sys.path.insert(0, '../jobs')

import json
from datetime import datetime, timezone

import joblib
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from config import (
    ARTIFACTS_DIR,
    REINTEGRATION_OUTCOME_MODEL_PATH, REINTEGRATION_OUTCOME_METADATA_PATH, REINTEGRATION_OUTCOME_METRICS_PATH,
    REINTEGRATION_TYPE_MODEL_PATH, REINTEGRATION_TYPE_METADATA_PATH, REINTEGRATION_TYPE_METRICS_PATH,
    EDUCATION_MODEL_PATH, EDUCATION_METADATA_PATH, EDUCATION_METRICS_PATH,
    WAREHOUSE_SCHEMA,
)
from utils_db import get_engine

print('Imports loaded.')

## Load data

In [ ]:
engine = get_engine(WAREHOUSE_SCHEMA)
df = pd.read_sql('SELECT * FROM fact_resident_outcomes_ml', engine)
print(f'Loaded {len(df)} rows')

print(f'\nReintegration outcome:\n{df["reintegration_outcome_label"].value_counts().to_string()}')
print(f'\nReintegration type:\n{df["reintegration_type_label"].value_counts().to_string()}')
print(f'\nEducation completion:\n{df["edu_completion_label"].value_counts().to_string()}')

## Feature columns

In [ ]:
FEATURE_COLS = [
    'age_upon_admission_years', 'length_of_stay_days', 'sex_encoded',
    'case_category_encoded', 'initial_risk_encoded', 'referral_source_encoded',
    'is_pwd', 'has_special_needs',
    'sub_cat_trafficked', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse',
    'sub_cat_child_labor', 'sub_cat_orphaned', 'sub_cat_at_risk',
    'family_is_4ps', 'family_solo_parent', 'family_indigenous',
    'avg_health_score', 'avg_nutrition', 'avg_sleep', 'avg_bmi',
    'pct_medical_done', 'pct_psych_done',
    'avg_attendance', 'avg_progress', 'num_edu_records',
    'total_sessions', 'pct_concerns_flagged', 'pct_referral_made', 'pct_progress_noted',
    'total_incidents', 'high_severity_count', 'pct_unresolved',
    'total_visits', 'pct_safety_concerns', 'avg_cooperation',
]
available = [c for c in FEATURE_COLS if c in df.columns]
print(f'{len(available)} feature columns available.')

## Model 1 — Reintegration outcome classifier

In [ ]:
df_o = df.dropna(subset=['reintegration_outcome_label'])
X_o  = df_o[available]
y_o  = df_o['reintegration_outcome_label']

min_class = y_o.value_counts().min()
stratify  = y_o if min_class >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    X_o, y_o, test_size=0.25, random_state=42, stratify=stratify
)

outcome_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')),
])
outcome_pipeline.fit(X_train, y_train)

y_pred   = outcome_pipeline.predict(X_test)
accuracy = float(accuracy_score(y_test, y_pred))
f1       = float(f1_score(y_test, y_pred, average='weighted', zero_division=0))
report   = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

print(f'Outcome — Accuracy: {accuracy:.3f} | Weighted F1: {f1:.3f}')
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(outcome_pipeline, str(REINTEGRATION_OUTCOME_MODEL_PATH))
with open(REINTEGRATION_OUTCOME_METADATA_PATH, 'w') as f:
    json.dump({'model_name': 'reintegration_outcome_classifier', 'model_version': '1.0.0',
               'trained_at_utc': datetime.now(timezone.utc).isoformat(),
               'warehouse_table': 'fact_resident_outcomes_ml',
               'num_training_rows': int(len(X_train)), 'num_test_rows': int(len(X_test)),
               'label_col': 'reintegration_outcome_label', 'feature_cols': available,
               'classes': list(outcome_pipeline.classes_)}, f, indent=2)
with open(REINTEGRATION_OUTCOME_METRICS_PATH, 'w') as f:
    json.dump({'accuracy': accuracy, 'weighted_f1': f1, 'classification_report': report}, f, indent=2)
print(f'Saved: {REINTEGRATION_OUTCOME_MODEL_PATH}')

## Model 2 — Reintegration type classifier

In [ ]:
df_t = df.dropna(subset=['reintegration_type_label'])
df_t = df_t[~df_t['reintegration_type_label'].isin(['None', ''])]
X_t  = df_t[available]
y_t  = df_t['reintegration_type_label']

min_class = y_t.value_counts().min()
stratify  = y_t if min_class >= 2 else None
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_t, y_t, test_size=0.25, random_state=42, stratify=stratify
)

type_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     GradientBoostingClassifier(n_estimators=100, random_state=42)),
])
type_pipeline.fit(X_train_t, y_train_t)

y_pred_t   = type_pipeline.predict(X_test_t)
accuracy_t = float(accuracy_score(y_test_t, y_pred_t))
f1_t       = float(f1_score(y_test_t, y_pred_t, average='weighted', zero_division=0))
report_t   = classification_report(y_test_t, y_pred_t, output_dict=True, zero_division=0)

print(f'Type — Accuracy: {accuracy_t:.3f} | Weighted F1: {f1_t:.3f}')
print(classification_report(y_test_t, y_pred_t, zero_division=0))

In [ ]:
joblib.dump(type_pipeline, str(REINTEGRATION_TYPE_MODEL_PATH))
with open(REINTEGRATION_TYPE_METADATA_PATH, 'w') as f:
    json.dump({'model_name': 'reintegration_type_classifier', 'model_version': '1.0.0',
               'trained_at_utc': datetime.now(timezone.utc).isoformat(),
               'warehouse_table': 'fact_resident_outcomes_ml',
               'num_training_rows': int(len(X_train_t)), 'num_test_rows': int(len(X_test_t)),
               'label_col': 'reintegration_type_label', 'feature_cols': available,
               'classes': list(type_pipeline.classes_)}, f, indent=2)
with open(REINTEGRATION_TYPE_METRICS_PATH, 'w') as f:
    json.dump({'accuracy': accuracy_t, 'weighted_f1': f1_t, 'classification_report': report_t}, f, indent=2)
print(f'Saved: {REINTEGRATION_TYPE_MODEL_PATH}')

## Model 3 — Education completion classifier

In [ ]:
df_e = df.dropna(subset=['edu_completion_label']).copy()
df_e['edu_completion_label'] = df_e['edu_completion_label'].str.replace(r'^Progress:\s*', '', regex=True)
X_e  = df_e[available]
y_e  = df_e['edu_completion_label']

min_class = y_e.value_counts().min()
stratify  = y_e if min_class >= 2 else None
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_e, y_e, test_size=0.25, random_state=42, stratify=stratify
)

edu_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')),
])
edu_pipeline.fit(X_train_e, y_train_e)

y_pred_e   = edu_pipeline.predict(X_test_e)
accuracy_e = float(accuracy_score(y_test_e, y_pred_e))
f1_e       = float(f1_score(y_test_e, y_pred_e, average='weighted', zero_division=0))
report_e   = classification_report(y_test_e, y_pred_e, output_dict=True, zero_division=0)

print(f'Education — Accuracy: {accuracy_e:.3f} | Weighted F1: {f1_e:.3f}')
print(classification_report(y_test_e, y_pred_e, zero_division=0))

In [ ]:
joblib.dump(edu_pipeline, str(EDUCATION_MODEL_PATH))
with open(EDUCATION_METADATA_PATH, 'w') as f:
    json.dump({'model_name': 'education_completion_classifier', 'model_version': '1.0.0',
               'trained_at_utc': datetime.now(timezone.utc).isoformat(),
               'warehouse_table': 'fact_resident_outcomes_ml',
               'num_training_rows': int(len(X_train_e)), 'num_test_rows': int(len(X_test_e)),
               'label_col': 'edu_completion_label', 'feature_cols': available,
               'classes': list(edu_pipeline.classes_)}, f, indent=2)
with open(EDUCATION_METRICS_PATH, 'w') as f:
    json.dump({'accuracy': accuracy_e, 'weighted_f1': f1_e, 'classification_report': report_e}, f, indent=2)
print(f'Saved: {EDUCATION_MODEL_PATH}')